# Worlds, runtime allocation, plain mode, and dispatch

This standalone lesson uses base DRYML and the Python standard library. It replaces the old compute-context lesson with current environment, world, runtime, and dispatch boundaries. It runs offline and creates all worker state in a temporary directory.

Environment values describe software compatibility. World values describe requested resources and topology. Both can be context-local planning defaults, but neither is an allocation. The active process allocation is reported by `dryml.runtime.active_runtime().allocation`.

In [ ]:
import operator
import sys
from tempfile import TemporaryDirectory

import dryml
from dryml.core2 import Definition
from dryml.core2.object import Object
from dryml.core2.store.dir import DirStore
from dryml.dispatch.errors import DispatchPlanningError
from dryml.environments import CurrentEnvironmentSpec, PythonExecutableSpec
from dryml.runtime.allocation import is_no_allocation
from dryml.worlds import LocalResourceInventory, WorldSpec


class PlainCounter(Object):
    def __init__(self, value):
        super().__init__()
        self.value = value

    def plus(self, value):
        return self.value + value


## Requested defaults are not active resources

The requested one-CPU world below is available to later planning. Entering environment and world scopes does not allocate that CPU or change this orchestrator's runtime identity.

In [ ]:
requested_world = WorldSpec.from_data({
    'roles': {
        'main': {
            'replicas': 1,
            'process': {'resources': {'cpus': 1}},
        }
    }
})
before_environment = dryml.environments.current()
before_world = dryml.worlds.current()
before_runtime = dryml.runtime.active_runtime()
assert is_no_allocation(before_runtime.allocation)

with dryml.environments.use(CurrentEnvironmentSpec()), dryml.worlds.use(requested_world):
    assert dryml.environments.current().kind == 'current'
    assert dryml.worlds.current() == requested_world
    assert dryml.runtime.active_runtime() is before_runtime
    assert is_no_allocation(dryml.runtime.active_runtime().allocation)

assert dryml.environments.current() is before_environment
assert dryml.worlds.current() is before_world
assert dryml.runtime.active_runtime() is before_runtime


## Trusted inline work with `runtime.plain()`

Plain mode is for trusted local work in the current process. It enters inline mode with a local allocation and enforcement off. It does not launch a worker, isolate code, or change the current environment and world defaults. The exact prior runtime object is restored after normal exit and after an exception.

In [ ]:
plain_before = dryml.runtime.active_runtime()
with dryml.runtime.plain() as inline_runtime:
    assert inline_runtime is dryml.runtime.active_runtime()
    assert inline_runtime.mode is dryml.runtime.RuntimeMode.INLINE
    assert not is_no_allocation(inline_runtime.allocation)
    assert inline_runtime.allocation.role == 'local'
    counter = Definition(PlainCounter, 3).build()
    assert counter.plus(4) == 7
assert dryml.runtime.active_runtime() is plain_before

try:
    with dryml.runtime.plain():
        raise RuntimeError('handled demonstration failure')
except RuntimeError as exc:
    assert str(exc) == 'handled demonstration failure'
assert dryml.runtime.active_runtime() is plain_before


## Explain first, then run a portable worker target

`dispatch.explain()` follows dispatch planning without launching a workload, activating an allocation, or writing Store records. Full explanations contain request-specific details, so stable examples inspect a small public summary.

For worker execution, `operator.add` has a standard-library import path. The explicit environment selects this Python executable with `pythonpath_policy='none'`: the worker receives no tutorial support module or added repository path. The temporary Store owns launch records and is deleted with the temporary directory.

In [ ]:
inventory = LocalResourceInventory((0,))
worker_environment = PythonExecutableSpec(
    sys.executable,
    pythonpath_policy='none',
).to_data()
dispatch_before = dryml.runtime.active_runtime()

with TemporaryDirectory(prefix='dryml-runtime-tutorial-') as directory:
    store = DirStore(directory, query_index='none')
    with dryml.environments.use(CurrentEnvironmentSpec()), dryml.worlds.use(requested_world):
        first = dryml.dispatch.explain(
            operator.add,
            store=store,
            inventory=inventory,
            args=(20, 22),
        )
        second = dryml.dispatch.explain(
            operator.add,
            store=store,
            inventory=inventory,
            args=(20, 22),
        )
        first_summary = {
            'launchable': first.launchable,
            'environment_source': first.resolution.environment_selection.source,
            'world_source': first.resolution.world_selection.source,
            'diagnostic_count': len(first.resolution.diagnostics),
        }
        second_summary = {
            'launchable': second.launchable,
            'environment_source': second.resolution.environment_selection.source,
            'world_source': second.resolution.world_selection.source,
            'diagnostic_count': len(second.resolution.diagnostics),
        }
        assert first_summary == second_summary
        assert first_summary['launchable'] is True
        assert first_summary['environment_source'] == 'current'
        assert first_summary['world_source'] == 'current'
        assert isinstance(first_summary['diagnostic_count'], int)
        assert 0 <= first_summary['diagnostic_count'] <= 256
        assert first.resolution.world_allocation_summary is None
        assert not store.records.records_dir.exists()
        assert dryml.runtime.active_runtime() is dispatch_before
        assert is_no_allocation(dryml.runtime.active_runtime().allocation)

        result = dryml.dispatch.run(
            operator.add,
            store=store,
            environment=worker_environment,
            inventory=inventory,
            args=(20, 22),
        )
        assert result.status == 'ok'
        assert result.result_canonical == 42
        assert dryml.runtime.active_runtime() is dispatch_before
        assert is_no_allocation(dryml.runtime.active_runtime().allocation)
    store.close()

assert dryml.runtime.active_runtime() is dispatch_before


## Notebook-local callables are not the portable worker path

A nested callable has no import path that a fresh worker can reproduce. DRYML rejects it by default and reports that `allow_pickle=True` is the explicit alternative. That option uses same-Python, same-environment pickle transport; it is non-portable and is not the recommended tutorial path, so this lesson does not execute it. Use an installed module-level function for portable worker dispatch.

In [ ]:
def make_notebook_local_target():
    def notebook_local_add(left, right):
        return left + right

    return notebook_local_add


notebook_local_add = make_notebook_local_target()
try:
    dryml.dispatch.explain(notebook_local_add, args=(1, 2))
except DispatchPlanningError as exc:
    assert exc.context['reason'] == 'local_function'
    assert exc.context['allow_pickle'] is True
    assert exc.context['transport'] == 'pickle_small'
    assert exc.context['transport_restrictions'] == ['same_python', 'same_environment']
else:
    raise AssertionError('a notebook-local target must require explicit pickle transport')


The three execution choices are intentionally distinct: trusted inline work uses `runtime.plain()` in this process; portable worker work uses an importable target such as `operator.add`; explicit pickle transport is a constrained same-Python fallback, not a portability mechanism. See `docs/world_runtime.md`, `docs/dispatch.md`, and `docs/migration/legacy_context_execute_removal.md` for the maintained contracts.